# Clickhouse

Clickhouse is an open-source OLAP DBMS. This DBMS specialises in processing large ammounts of data and is primarily used by analytics and ML teams to extract data from huge datasets.

## Describe

Use the `TABLE DESCRIBE` command to view the properties of the table. This is practcally useful for identifing the names and types of the columns.

Check more in the page [DESCRIBE TABLE](https://clickhouse.com/docs/sql-reference/statements/describe-table).

---

The following cell shows the example of `DESCRIBE TABLE` usage.

In [3]:
--ClickHouse
DESCRIBE TABLE
SELECT *
FROM VALUES(
    'id UInt32, name String',
    (1, 'Alice'),
    (2, 'Bob'),
    (3, 'Carol')
);

name,type,default_type,default_expression,comment,codec_expression,ttl_expression
id,UInt32,,,,,
name,String,,,,,


## Databases

Most essences in clickhouse are separated by databases. The database can contain: tables, views, dictionaries and functions.

Important commands to handle databases:

- `SHOW DATABASES`.
- `CREATE DATABASE <name>`
- `DROP DATABASE <name>`.
- `USE DATABASE`: to select the context, all subsequent queries will automatilly will refer to the essesnces of the specific database.

---

The following cell show the initial list of available databases.

In [2]:
--ClickHouse
SHOW DATABASES;

name
INFORMATION_SCHEMA
default
information_schema
system


After `CREATE DATABASE` there is a new database with the corresponding name:

In [5]:
--ClickHouse
CREATE DATABASE test_database;
SHOW DATABASES;

name
INFORMATION_SCHEMA
default
information_schema
system
test_database


After `DROP DATABASE` the database is removed.

In [6]:
--ClickHouse
DROP DATABASE test_database;
SHOW DATABASES;

name
INFORMATION_SCHEMA
default
information_schema
system


## Parts

The clickhouse stores data as parts. It manipulates these parts over time. Information about the current parts is stored in the `system.parts` table.

The practically important columns for the `system.parts` table are:

- `name`: the name of the part, there is encoded some iformation about the part.
- `table`: describes to which table part is belongs to.
- `active`: colum takes value 1 if the corresponding part actually is used.

---

The following cell displays the description for the `system.parts` table.

In [2]:
--ClickHouse
DESCRIBE TABLE system.parts;

name,type,default_type,default_expression,comment,codec_expression,ttl_expression
partition,String,,,The partition name.,,
name,String,,,"Name of the data part. The part naming structure can be used to determine many aspects of the data, ingest, and merge patterns. The part naming format is the following: ``` <partition_id>_<minimum_block_number>_<maximum_block_number>_<level>_<data_version> ``` * Definitions: - `partition_id` - identifies the partition key - `minimum_block_number` - identifies the minimum block number in the part. ClickHouse always merges continuous blocks - `maximum_block_number` - identifies the maximum block number in the part - `level` - incremented by one with each additional merge on the part. A level of 0 indicates this is a new part that has not been merged. It is important to remember that all parts in ClickHouse are always immutable - `data_version` - optional value, incremented when a part is mutated (again, mutated data is always only written to a new part, since parts are immutable)",,
uuid,UUID,,,The UUID of data part.,,
part_type,String,,,The data part storing format. Possible Values: Wide (a file per column) and Compact (a single file for all columns).,,
active,UInt8,,,"Flag that indicates whether the data part is active. If a data part is active, it's used in a table. Otherwise, it's about to be deleted. Inactive data parts appear after merging and mutating operations.",,
marks,UInt64,,,"The number of marks. To get the approximate number of rows in a data part, multiply marks by the index granularity (usually 8192) (this hint does not work for adaptive granularity).",,
rows,UInt64,,,The number of rows.,,
files,UInt64,,,The number of files in the data part.,,
bytes_on_disk,UInt64,,,Total size of all the data part files in bytes.,,
data_compressed_bytes,UInt64,,,"Total size of compressed data in the data part. All the auxiliary files (for example, files with marks) are not included.",,
